# 02 – Calculating peptide features

**Input:** `results/clean_peptides.csv` (from notebook 01)  
**Output:** `results/peptide_features.csv` – one row per peptide with a handful of simple, easy-to-explain features.

**Features** (all computed from the amino-acid sequence only):

| Feature | Definition |
|---|---|
| `length` | number of residues (already in the data) |
| `net_charge` | (# K + # R) − (# D + # E). Histidine is ignored (it is only partly charged at neutral pH). |
| `charge_density` | `net_charge / length`. Natural peptides are longer, so raw charge alone would be unfair to compare. |
| `hydrophobic_fraction` | share of residues that are A, V, L, I, M, F or W |
| `pct_A ... pct_Y` | percentage of each of the 20 amino acids |

Each feature is small enough to compute by hand, which helps when explaining the method in a report.

In [1]:
import pandas as pd

IN = "../results/clean_peptides.csv"
OUT = "../results/peptide_features.csv"

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
POSITIVE = set("KR")
NEGATIVE = set("DE")
HYDROPHOBIC = set("AVLIMFW")

clean = pd.read_csv(IN)
print(clean.shape)
clean.head(3)

(18196, 24)


,ID,name,sequence,length,length_class,origin,synthesis_type,kingdom,kingdom_known,has_D_aa,...,is_cancer,is_virus,is_biofilm,is_parasite,is_insect,is_nematode,is_mollicute,is_protista,is_archaea,is_mammalian_cell
0,11,"Cathelicidin-1, CATH-1, Fowlicidin-1",RVKRVWPLVIRTVIAGYNLYRAIKKK,26,Medium (25–50 aa),Natural,Ribosomal,Animalia,True,False,...,1,0,0,0,0,0,0,0,0,1
1,14,Brevinin-2 related peptide B2RP,GIWDTIKSMGKVFAGKILQNL,21,Short (10–24 aa),Natural,Ribosomal,Animalia,True,False,...,0,1,0,0,0,0,0,0,0,1
2,15,Histatin 5,DSHAKRHHGYKRKFHEKHHSHRGY,24,Short (10–24 aa),Natural,Ribosomal,Animalia,True,False,...,0,0,0,1,0,0,0,0,0,1


## Feature functions
Written as plain small functions and tested on toy sequences before running on the real data.

In [2]:
def net_charge(seq):
    return sum(aa in POSITIVE for aa in seq) - sum(aa in NEGATIVE for aa in seq)

def hydrophobic_fraction(seq):
    return sum(aa in HYDROPHOBIC for aa in seq) / len(seq)

def composition(seq):
    """Percentage of each amino acid, as a dict like {'pct_A': 10.0, ...}"""
    return {f"pct_{aa}": 100 * seq.count(aa) / len(seq) for aa in AMINO_ACIDS}

# --- quick tests on toy sequences (worked out by hand) ---
assert net_charge("KKRDE") == 1           # 3 positive - 2 negative
assert net_charge("GGGG") == 0
assert net_charge("DDEE") == -4
assert hydrophobic_fraction("AVLG") == 0.75   # A, V, L hydrophobic; G not
assert hydrophobic_fraction("GSTN") == 0
assert composition("AAGG")["pct_A"] == 50 and composition("AAGG")["pct_K"] == 0
assert abs(sum(composition("ACDEFGHIKL").values()) - 100) < 1e-9
print("toy tests passed")

toy tests passed


## Compute features

In [3]:
feat = clean[["ID", "name", "sequence", "origin", "synthesis_type", "kingdom", "kingdom_known", "length", "length_class",
              "has_D_aa", "has_target_info"]].copy()
target_cols = [c for c in clean.columns if c.startswith("is_")]
feat[target_cols] = clean[target_cols]

feat["net_charge"] = clean["sequence"].apply(net_charge)
feat["charge_density"] = feat["net_charge"] / feat["length"]
feat["hydrophobic_fraction"] = clean["sequence"].apply(hydrophobic_fraction)

comp = pd.DataFrame(clean["sequence"].apply(composition).tolist())
feat = pd.concat([feat, comp], axis=1)
feat.shape

(18196, 47)

## Sanity checks

In [4]:
assert len(feat) == len(clean) and feat["ID"].is_unique
assert feat[["net_charge", "charge_density", "hydrophobic_fraction"]].notna().all().all()
assert feat["hydrophobic_fraction"].between(0, 1).all()
assert (feat[[f"pct_{aa}" for aa in AMINO_ACIDS]].sum(axis=1) - 100).abs().max() < 1e-6
# net charge can never be bigger than the sequence length
assert (feat["net_charge"].abs() <= feat["length"]).all()
print("all checks passed")

all checks passed


## First look: Natural vs Synthetic
Just medians (no plots or tests yet – that is notebook 03). This is only to see that the numbers make sense: antimicrobial peptides are expected to be positively charged and fairly hydrophobic.

In [5]:
cols = ["length", "net_charge", "charge_density", "hydrophobic_fraction"]
feat.groupby("origin")[cols].median().round(2)

,length,net_charge,charge_density,hydrophobic_fraction
origin,,,,
Natural,25.0,3.0,0.10,0.42
Synthetic,15.0,4.0,0.29,0.44


In [6]:
# Amino acids with the biggest difference in mean percentage between Natural and Synthetic
means = feat.groupby("origin")[[f"pct_{aa}" for aa in AMINO_ACIDS]].mean().T
means["diff (Nat - Syn)"] = means["Natural"] - means["Synthetic"]
means.round(2).sort_values("diff (Nat - Syn)", key=abs, ascending=False).head(6)

origin,Natural,Synthetic,diff (Nat - Syn)
pct_K,9.66,17.73,-8.07
pct_R,5.07,12.91,-7.84
pct_G,10.75,5.51,5.24
pct_W,1.80,6.60,-4.80
pct_C,5.36,2.10,3.26
pct_S,5.28,2.88,2.40


## Save

In [7]:
feat.to_csv(OUT, index=False)
print("saved", feat.shape, "->", OUT)

saved (18196, 47) -> ../results/peptide_features.csv


## Notes for the methods section
* Net charge counts K/R (+1) and D/E (−1) only; N/C-terminus modifications (e.g. amidation) and histidine are ignored for simplicity.
* Charge density is included next to raw net charge because length differs strongly between groups.
* The hydrophobic set (A, V, L, I, M, F, W) is one common simple definition; results can shift slightly with other scales.
* Features are computed on L-letter sequences: D-amino acids were converted to their L-letter in notebook 01 (`has_D_aa` flag kept).